# 16 · Convertibles, Inflation, and TRS Hybrids

Hybrid instruments mix fixed income, equity, and inflation state variables. This notebook covers convertible bonds alongside inflation-linked trades.

### Learning Objectives
- Configure convertible bonds with conversion policies and volatility surfaces
- Price inflation-linked bonds and zero-coupon inflation swaps
- Use the same market context for hybrids

In [ ]:
from datetime import date

from finstack import Money
from finstack.core.currency import USD
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.scalars import MarketScalar
from finstack.core.market_data.surfaces import VolSurface
from finstack.core.market_data.term_structures import DiscountCurve, InflationCurve
from finstack.valuations.cashflow import FixedCouponSpec, ScheduleParams, CouponType
from finstack.valuations.instruments import (
    ConvertibleBond,
    ConversionEvent,
    ConversionPolicy,
    ConversionSpec,
    InflationLinkedBond,
    InflationSwap,
)
from finstack.valuations.pricer import create_standard_registry

as_of = date(2024, 1, 2)
market = MarketContext()
market.insert_discount(
    DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.9970), (1.0, 0.9940), (3.0, 0.9730), (5.0, 0.9490)],
    )
)
market.insert_price("EQUITY-SPOT", MarketScalar.price(Money(150.0, USD)))
market.insert_price("EQUITY-DIVYIELD", MarketScalar.unitless(0.015))
market.insert_surface(
    VolSurface(
        "EQUITY-VOL",
        expiries=[0.25, 0.5, 1.0, 2.0],
        strikes=[120.0, 140.0, 160.0, 180.0],
        grid=[
            [0.28, 0.26, 0.25, 0.24],
            [0.27, 0.25, 0.24, 0.23],
            [0.26, 0.24, 0.23, 0.22],
            [0.25, 0.23, 0.22, 0.21],
        ],
    )
)
market.insert_inflation(
    InflationCurve(
        "US-CPI",
        base_cpi=300.0,
        knots=[(0.0, 300.0), (1.0, 303.0), (2.0, 306.5), (5.0, 320.0), (10.0, 345.0)],
    )
)
registry = create_standard_registry()


## 1. Convertible Bond
Use conversion events/policies to define equity optionality. Here we add a price-trigger conversion ratio.

In [ ]:
issue = as_of
maturity = date(2029, 1, 2)
schedule = ScheduleParams.semiannual_30360()
fixed_coupon = FixedCouponSpec.new(0.035, schedule, CouponType.CASH)
policy = ConversionPolicy.upon_event(ConversionEvent.price_trigger(160.0, 30))
conversion_spec = ConversionSpec.create(policy, ratio=20.0, anti_dilution=None)
convertible = ConvertibleBond.create(
    "ACME-CB",
    Money(1_000_000, USD),
    issue,
    maturity,
    "USD-OIS",
    conversion_spec,
    underlying_equity_id="EQUITY-SPOT",
    call_schedule=[(date(2027, 1, 2), 102.5)],
    fixed_coupon=fixed_coupon,
)
conv_res = registry.price_with_metrics(convertible, "discounting", market, ["delta", "gamma", "vega"])
print("Convertible PV:", conv_res.value)
print("Delta:", conv_res.measures.get("delta"))


## 2. Inflation-Linked Bond
Link coupon/principal to CPI via the inflation curve.

In [ ]:
ilb = InflationLinkedBond.create(
    "US-TIPS-2033",
    Money(1_000_000, USD),
    real_coupon=0.0125,
    issue=as_of,
    maturity=date(2034, 1, 15),
    base_index=300.0,
    discount_curve="USD-OIS",
    inflation_curve="US-CPI",
    indexation="tips",
)
ilb_res = registry.price_with_metrics(ilb, "discounting", market, ["real_duration", "breakeven_inflation"])
print("ILB PV:", ilb_res.value)
print("Real duration:", ilb_res.measures.get("real_duration"))
print("Breakeven inflation:", ilb_res.measures.get("breakeven_inflation"))


## 3. Zero-Coupon Inflation Swap
Inflation swaps exchange fixed inflation expectations versus realized CPI.

In [ ]:
inf_swap = InflationSwap.create(
    "US-ZC-INFLATION-SWAP",
    Money(5_000_000, USD),
    fixed_rate=0.025,
    start_date=as_of,
    maturity=date(2030, 1, 2),
    discount_curve="USD-OIS",
    inflation_curve="US-CPI",
    side="pay_fixed",
)
swap_res = registry.price_with_metrics(inf_swap, "discounting", market, ["par_rate", "npv01"])
print("Inflation swap PV:", swap_res.value)
print("Par rate:", swap_res.measures.get("par_rate"))
print("NPV01:", swap_res.measures.get("npv01"))


## Summary
- Convertible bonds use conversion policies to tie equity option value into debt cashflows.
- Inflation-linked securities share CPI curves and produce real-duration metrics.
- Hybrids fit naturally into the same `MarketContext` and metric infrastructure as vanilla bonds.